In [ ]:
# ── CELL 0: Imports ────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)
print('Ready.')

In [ ]:
# ── CELL 1: Derivatives — Slope at a Point ─────────────────────────────────────
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

f = lambda w: w**2   # L(w) = w^2  (simplest bowl-shaped loss)

w = 3.0
print(f'f(w) = w^2 at w={w}:')
print(f'  Numerical derivative: {numerical_derivative(f, w):.6f}')
print(f'  True derivative 2w:   {2*w:.6f}')

# Derivative sign tells you which way to move
w_vals = np.linspace(-3, 3, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(w_vals, w_vals**2, 'steelblue', lw=2, label='L(w) = w²')
w0 = 2.0
tangent = 2*w0*(w_vals - w0) + w0**2
axes[0].plot(w_vals, tangent, 'tomato', ls='--', label=f'Tangent at w=2 (slope=4)')
axes[0].scatter([w0], [w0**2], color='tomato', s=100, zorder=5)
axes[0].set_title('L(w) = w²: derivative = slope of tangent')
axes[0].set_ylim(-1, 10); axes[0].legend()

axes[1].plot(w_vals, 2*w_vals, 'seagreen', lw=2)
axes[1].axhline(0, color='k', ls='--', alpha=0.5)
axes[1].axvline(0, color='k', ls='--', alpha=0.5, label='minimum: gradient=0')
axes[1].set_title('dL/dw = 2w'); axes[1].legend()
plt.tight_layout(); plt.show()

print('\nDerivative sign -> which way to move weight:')
for w_t in [-2.0, 0.0, 2.0]:
    d = 2*w_t
    move = 'move LEFT' if d > 0 else ('move RIGHT' if d < 0 else 'AT MINIMUM')
    print(f'  w={w_t:5.1f}  dL/dw={d:5.1f}  -> {move}')

In [ ]:
# ── CELL 2: Partial Derivatives + Gradient Descent in 2D ──────────────────────
# L(w1, w2) = w1^2 + 2*w2^2
# dL/dw1 = 2*w1,  dL/dw2 = 4*w2

def loss_fn(w1, w2):  return w1**2 + 2*w2**2
def grad_fn(w):       return np.array([2*w[0], 4*w[1]])

w = np.array([2.0, 1.5])
lr = 0.1
trajectory = [w.copy()]

for _ in range(25):
    g  = grad_fn(w)
    w  = w - lr * g
    trajectory.append(w.copy())

trajectory = np.array(trajectory)

print(f'Gradient descent on L(w1,w2) = w1² + 2w2²')
for i in [0, 5, 10, 24]:
    print(f'  step {i:2d}: w={trajectory[i]}  L={loss_fn(*trajectory[i]):.6f}')

w1g = np.linspace(-2.5, 2.5, 80); w2g = np.linspace(-2, 2, 80)
W1, W2 = np.meshgrid(w1g, w2g)

fig, ax = plt.subplots(figsize=(8, 6))
cp = ax.contourf(W1, W2, loss_fn(W1, W2), levels=30, cmap='viridis')
ax.plot(trajectory[:,0], trajectory[:,1], 'r.-', lw=2, ms=8)
ax.scatter(*trajectory[0],  color='red',  s=100, zorder=5, label='start')
ax.scatter(*trajectory[-1], color='lime', s=100, zorder=5, label='end (minimum)')
ax.set_title('Gradient Descent — Contour View')
ax.set_xlabel('w1'); ax.set_ylabel('w2'); ax.legend()
plt.colorbar(cp, label='Loss')
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 3: Chain Rule — The Core of Backpropagation ──────────────────────────
# x -> z1=w1*x -> relu -> a1 -> z2=w2*a1 -> L=z2^2

x, w1, w2 = 2.0, 0.5, 0.3

# Forward pass
z1 = w1 * x
a1 = max(0, z1)
z2 = w2 * a1
L  = z2**2

print('Forward pass:')
print(f'  z1={z1}, a1={a1}, z2={z2:.4f}, L={L:.4f}')

# Backward pass (chain rule)
dL_dz2 = 2 * z2
dz2_da1 = w2
da1_dz1 = 1.0 if z1 > 0 else 0.0
dz1_dw1 = x

dL_dw1 = dL_dz2 * dz2_da1 * da1_dz1 * dz1_dw1
dL_dw2 = dL_dz2 * a1

print('\nBackward pass (chain rule):')
print(f'  dL/dz2={dL_dz2:.4f}  dz2/da1={dz2_da1}  da1/dz1={da1_dz1}  dz1/dw1={dz1_dw1}')
print(f'  dL/dw1 = {dL_dz2:.4f} * {dz2_da1} * {da1_dz1} * {dz1_dw1} = {dL_dw1:.4f}')
print(f'  dL/dw2 = {dL_dz2:.4f} * {a1} = {dL_dw2:.4f}')

# Verify numerically
h = 1e-5
def fwd(w1v, w2v):
    z1v = w1v * x; a1v = max(0, z1v); z2v = w2v * a1v; return z2v**2

num_dw1 = (fwd(w1+h, w2) - fwd(w1-h, w2)) / (2*h)
num_dw2 = (fwd(w1, w2+h) - fwd(w1, w2-h)) / (2*h)
print(f'\nNumerical verify: dL/dw1={num_dw1:.6f}  dL/dw2={num_dw2:.6f}')
print(f'Analytical:       dL/dw1={dL_dw1:.6f}  dL/dw2={dL_dw2:.6f}')
print(f'Match: {np.isclose(dL_dw1, num_dw1) and np.isclose(dL_dw2, num_dw2)}')

lr = 0.1
print(f'\nWeight update: w1: {w1} -> {w1 - lr*dL_dw1:.4f}  w2: {w2} -> {w2 - lr*dL_dw2:.4f}')

In [ ]:
# ── CELL 4: Gradient Descent Variants — Batch vs SGD vs Mini-batch ─────────────
np.random.seed(42)
n = 200
X_d = np.random.randn(n, 1)
y_d = 3 * X_d.flatten() + 1.5 + np.random.randn(n) * 0.5

def mse(w, b):    return np.mean((y_d - (w*X_d.flatten()+b))**2)
def grads(w, b):  e = (w*X_d.flatten()+b) - y_d; return 2*np.mean(e*X_d.flatten()), 2*np.mean(e)

# Batch GD
w, b = 0.0, 0.0; bgd_losses = []
for _ in range(100): bgd_losses.append(mse(w,b)); dw,db = grads(w,b); w-=0.1*dw; b-=0.1*db
print(f'Batch GD:      w={w:.4f} b={b:.4f}  (true: 3.0, 1.5)')

# SGD (1 sample per step)
w, b = 0.0, 0.0; sgd_losses = []
for ep in range(10):
    for i in np.random.permutation(n):
        e = w*X_d[i,0]+b - y_d[i]; w -= 0.01*2*e*X_d[i,0]; b -= 0.01*2*e
    sgd_losses.append(mse(w,b))
print(f'SGD:           w={w:.4f} b={b:.4f}')

# Mini-batch SGD
w, b = 0.0, 0.0; mb_losses = []
for ep in range(10):
    perm = np.random.permutation(n)
    for start in range(0, n, 32):
        idx = perm[start:start+32]; Xb = X_d[idx,0]; yb = y_d[idx]
        e = w*Xb+b-yb; w -= 0.05*2*np.mean(e*Xb); b -= 0.05*2*np.mean(e)
    mb_losses.append(mse(w,b))
print(f'Mini-batch SGD: w={w:.4f} b={b:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(bgd_losses, 'steelblue', lw=2, label='Batch GD')
axes[0].plot(sgd_losses, 'tomato',    lw=2, label='SGD')
axes[0].plot(mb_losses,  'seagreen',  lw=2, label='Mini-batch SGD')
axes[0].set_title('Convergence Comparison'); axes[0].set_xlabel('Epoch'); axes[0].legend()

for lr_t, c in [(0.001,'steelblue'),(0.05,'seagreen'),(0.1,'orange'),(0.5,'tomato')]:
    wt, bt = 0.0, 0.0; lt = []
    for _ in range(50): lt.append(mse(wt,bt)); dw,db=grads(wt,bt); wt-=lr_t*dw; bt-=lr_t*db
    axes[1].plot(lt, label=f'lr={lr_t}', color=c, lw=2)
axes[1].set_title('Learning Rate Sensitivity'); axes[1].set_xlabel('Step'); axes[1].set_ylim(0,20); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 5: Activation Functions and Their Derivatives ─────────────────────────
z = np.linspace(-4, 4, 300)

sigmoid   = lambda z: 1/(1+np.exp(-z))
sigmoid_d = lambda z: sigmoid(z)*(1-sigmoid(z))      # max 0.25
tanh_d    = lambda z: 1 - np.tanh(z)**2              # max 1.0
relu      = lambda z: np.maximum(0, z)
relu_d    = lambda z: (z > 0).astype(float)           # 0 or 1
leaky     = lambda z: np.where(z>0, z, 0.01*z)
leaky_d   = lambda z: np.where(z>0, 1.0, 0.01)
gelu      = lambda z: z * sigmoid(1.702*z)
gelu_d    = lambda z: sigmoid(1.702*z) + z*1.702*sigmoid(1.702*z)*(1-sigmoid(1.702*z))

acts = [('Sigmoid', sigmoid, sigmoid_d, 'tomato'),
        ('Tanh', np.tanh, tanh_d, 'steelblue'),
        ('ReLU', relu, relu_d, 'seagreen'),
        ('Leaky ReLU', leaky, leaky_d, 'orange'),
        ('GELU', gelu, gelu_d, 'purple')]

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for col, (name, f, df, color) in enumerate(acts):
    axes[0,col].plot(z, f(z), color=color, lw=2)
    axes[0,col].set_title(f'{name}'); axes[0,col].axhline(0, color='k', alpha=0.3)
    axes[1,col].plot(z, df(z), color=color, lw=2, ls='--')
    axes[1,col].set_title(f"Derivative max={df(z).max():.2f}")
    axes[1,col].set_ylim(-0.1, 1.2)
plt.suptitle('Activations and Derivatives — Max derivative = gradient flow capacity', fontsize=12)
plt.tight_layout(); plt.show()

print('Vanishing gradient: sigmoid deriv max = 0.25')
print(f'  After 10 layers: 0.25^10 = {0.25**10:.2e}  (gradient dies)')
print('ReLU deriv = 1.0 -> gradients flow freely (2012: AlexNet breakthrough)')
print('GELU: smooth ReLU, used in BERT and GPT transformers')

In [ ]:
# ── CELL 6: Vanishing and Exploding Gradients ──────────────────────────────────
np.random.seed(42)

def grad_flow(n_layers, activation):
    grad = 1.0; history = [grad]
    for _ in range(n_layers):
        z = np.random.randn()
        if activation == 'sigmoid': s=1/(1+np.exp(-z)); ld = s*(1-s)
        elif activation == 'relu':  ld = 1.0 if z>0 else 0.0
        elif activation == 'leaky': ld = 1.0 if z>0 else 0.01
        ws = abs(np.random.randn() * 0.5)
        grad *= ld * ws; history.append(grad)
    return history

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for act, c in [('sigmoid','tomato'),('relu','steelblue'),('leaky','seagreen')]:
    axes[0].plot(grad_flow(30, act), label=act, color=c, lw=2)
axes[0].set_yscale('log'); axes[0].set_title('Gradient Magnitude (log scale)')
axes[0].set_xlabel('Layer (output->input)'); axes[0].legend()

# Exploding vs clipped
g1, g2 = 1.0, 1.0; clip=1.0; exp_t, clip_t = [g1],[g2]
for _ in range(30):
    ws = np.random.uniform(1.1, 1.3)
    g1 *= ws; g2 *= ws
    if abs(g2) > clip: g2 = np.sign(g2)*clip
    exp_t.append(g1); clip_t.append(g2)
axes[1].plot(exp_t,  'tomato',    lw=2, label='No clipping (explodes)')
axes[1].plot(clip_t, 'steelblue', lw=2, label='Gradient clipping (stable)')
axes[1].set_title('Exploding Gradients + Fix'); axes[1].legend()
plt.tight_layout(); plt.show()

print('FIXES for vanishing: ReLU, residual connections, batch norm, Xavier/He init')
print('FIXES for exploding: gradient clipping, lower lr, batch norm')

In [ ]:
# ── CELL 7: PyTorch Autograd — Chain Rule on Autopilot ─────────────────────────
# Scalar example
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)
z = w * x
L = z ** 2
L.backward()
print(f'Forward: w={w.item()}, x={x.item()}, z={z.item()}, L={L.item()}')
print(f'dL/dw = {w.grad.item()}  (= 2*z*x = 2*6*3 = 36)')

# Full training loop
np.random.seed(42); torch.manual_seed(42)
n_pts = 100
X_t = torch.FloatTensor(np.random.randn(n_pts, 1).astype(np.float32))
y_t = torch.FloatTensor((3*X_t.numpy() + 1.5 + 0.3*np.random.randn(n_pts,1)).astype(np.float32))

model     = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
criterion = nn.MSELoss()
losses    = []

for epoch in range(100):
    y_pred = model(X_t)            # forward: matrix multiply
    loss   = criterion(y_pred, y_t)
    optimizer.zero_grad()          # CRITICAL: clear old gradients
    loss.backward()                # chain rule -> fills .grad for all params
    optimizer.step()               # w = w - lr * w.grad
    losses.append(loss.item())

print(f'\nTrained: w={model.weight.item():.4f} (true=3.0)  b={model.bias.item():.4f} (true=1.5)')

# Gradient accumulation pitfall
print('\nPitfall: forgetting zero_grad() -> gradients accumulate:')
w2 = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    L2 = (w2*2)**2; L2.backward()
    print(f'  iter {i}: grad={w2.grad.item():.1f}  (should be same each time!)')

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(losses, 'steelblue', lw=2)
ax.set_title('PyTorch SGD Loss — Autograd handles all calculus')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 8: Adam Optimizer — Adaptive Learning Rates ──────────────────────────
def adam_step(w, g, m, v, t, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    m = b1*m + (1-b1)*g
    v = b2*v + (1-b2)*g**2
    mh = m/(1-b1**t)
    vh = v/(1-b2**t)
    return w - lr*mh/(np.sqrt(vh)+eps), m, v

# Rosenbrock function — notoriously hard for SGD
def rosen(x, y):    return (1-x)**2 + 100*(y-x**2)**2
def rosen_g(x, y):  return np.array([-2*(1-x)-400*x*(y-x**2), 200*(y-x**2)])

# SGD
w_s = np.array([-1.5, 1.5]); ts = [w_s.copy()]
for _ in range(5000): g=rosen_g(*w_s); w_s=w_s-0.001*g; ts.append(w_s.copy())
ts = np.array(ts)

# Adam
w_a = np.array([-1.5, 1.5]); m_a=np.zeros(2); v_a=np.zeros(2); ta=[w_a.copy()]
for t in range(1, 5001): g=rosen_g(*w_a); w_a,m_a,v_a=adam_step(w_a,g,m_a,v_a,t,lr=0.01); ta.append(w_a.copy())
ta = np.array(ta)

print(f'SGD  final: ({w_s[0]:.4f}, {w_s[1]:.4f})  target=(1,1)')
print(f'Adam final: ({w_a[0]:.4f}, {w_a[1]:.4f})  target=(1,1)')

xr=np.linspace(-2,2,200); yr=np.linspace(-0.5,2.5,200)
Xr,Yr=np.meshgrid(xr,yr); Zr=np.log(1+rosen(Xr,Yr))

fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,traj,name,color in [(axes[0],ts,'SGD','tomato'),(axes[1],ta,'Adam','steelblue')]:
    ax.contourf(Xr,Yr,Zr,levels=30,cmap='viridis',alpha=0.7)
    ax.plot(traj[:,0],traj[:,1],color=color,lw=1,alpha=0.7,label=name)
    ax.scatter(*traj[0],s=80,color='white',zorder=5,label='start')
    ax.scatter(*traj[-1],s=80,color='yellow',zorder=5,label='end')
    ax.scatter(1,1,s=80,color='lime',zorder=5,marker='*',label='minimum')
    ax.set_title(f'{name} — 5000 steps'); ax.legend(fontsize=8)
    ax.set_xlim(-2,2); ax.set_ylim(-0.5,2.5)
plt.suptitle('Adam reaches minimum, SGD struggles (Rosenbrock function)', fontsize=12)
plt.tight_layout(); plt.show()

print('\nAdam key params: beta1=0.9 (gradient momentum) beta2=0.999 (gradient variance)')
print('AdamW = Adam + weight decay -> standard for LLMs (BERT, GPT, etc.)')

In [ ]:
# ── CELL 9: Full PyTorch Training Loop — All Concepts Combined ─────────────────
torch.manual_seed(42); np.random.seed(42)

X_np, y_np = make_moons(n_samples=600, noise=0.2, random_state=42)
X_np = StandardScaler().fit_transform(X_np)
X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2)

X_train = torch.FloatTensor(X_tr)
y_train = torch.FloatTensor(y_tr).unsqueeze(1)
X_test  = torch.FloatTensor(X_te)
y_test  = torch.FloatTensor(y_te).unsqueeze(1)

model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid()
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

train_losses, test_losses, grad_norms = [], [], []

for epoch in range(200):
    model.train()
    y_pred = model(X_train)               # forward pass (matrix multiplies)
    loss   = criterion(y_pred, y_train)   # cross-entropy = -log-likelihood
    optimizer.zero_grad()
    loss.backward()                        # chain rule for ALL weights
    gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevent explosion
    grad_norms.append(gn)
    optimizer.step()                       # Adam weight update
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        test_losses.append(criterion(model(X_test), y_test).item())

model.eval()
with torch.no_grad():
    tr_acc = ((model(X_train)>0.5).float()==y_train).float().mean().item()
    te_acc = ((model(X_test) >0.5).float()==y_test ).float().mean().item()

print(f'Train acc: {tr_acc:.4f}  |  Test acc: {te_acc:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(train_losses, 'steelblue', label='Train'); axes[0].plot(test_losses, 'tomato', ls='--', label='Test')
axes[0].set_title('Loss (BCELoss = negative log-likelihood)'); axes[0].legend()

axes[1].plot(grad_norms, 'seagreen', alpha=0.7); axes[1].axhline(1.0, color='red', ls='--', label='clip threshold')
axes[1].set_title('Gradient Norms'); axes[1].legend()

xx,yy = np.meshgrid(np.linspace(-3,3,100),np.linspace(-3,3,100))
grid  = torch.FloatTensor(np.c_[xx.ravel(),yy.ravel()])
with torch.no_grad(): probs = model(grid).numpy().reshape(xx.shape)
axes[2].contourf(xx,yy,probs,levels=50,cmap='RdBu',alpha=0.7)
axes[2].scatter(X_te[:,0],X_te[:,1],c=y_te,cmap='RdBu',edgecolors='k',s=20)
axes[2].set_title(f'Decision Boundary (acc={te_acc:.3f})')
plt.tight_layout(); plt.show()

In [ ]:
# ── CELL 10: Practice — Diagnose and Fix Exploding Loss ───────────────────────
torch.manual_seed(42); np.random.seed(42)

n = 300
Xn = torch.FloatTensor(np.random.randn(n,4).astype(np.float32))
yn = torch.FloatTensor((Xn[:,0]*2 - Xn[:,1] + 0.5*Xn[:,2] + torch.randn(n)*0.3).numpy()).unsqueeze(1)

def build():  return nn.Sequential(nn.Linear(4,32),nn.Tanh(),nn.Linear(32,32),nn.Tanh(),nn.Linear(32,1))

def train_model(model, opt, clip=None, epochs=100):
    crit = nn.MSELoss(); losses = []; norms = []
    for _ in range(epochs):
        yp = model(Xn); l = crit(yp, yn)
        opt.zero_grad(); l.backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), clip if clip else 1e9)
        norms.append(gn); opt.step(); losses.append(l.item())
    return losses, norms

m1,o1 = build(), torch.optim.SGD(build().parameters(), lr=1.0)    # broken
m2,o2 = build(), torch.optim.SGD(build().parameters(), lr=0.01)   # fix 1: lower lr
m3,o3 = build(), torch.optim.SGD(build().parameters(), lr=0.1)    # fix 2: clipping
m4,o4 = build(), torch.optim.Adam(build().parameters(), lr=0.01)  # fix 3: Adam

l1,n1 = train_model(build(), torch.optim.SGD(build().parameters(), lr=1.0))
l2,n2 = train_model(build(), torch.optim.SGD(build().parameters(), lr=0.01))
l3,n3 = train_model(build(), torch.optim.SGD(build().parameters(), lr=0.1), clip=1.0)
l4,n4 = train_model(build(), torch.optim.Adam(build().parameters(), lr=0.01))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(l1,'tomato',   lw=2,label='SGD lr=1.0 (BROKEN)')
axes[0].plot(l2,'steelblue',lw=2,label='SGD lr=0.01 (fix: lower lr)')
axes[0].plot(l3,'orange',   lw=2,label='SGD+clip lr=0.1')
axes[0].plot(l4,'seagreen', lw=2,label='Adam lr=0.01')
axes[0].set_title('Loss Curves — Diagnosing Exploding Loss'); axes[0].set_ylim(0,30); axes[0].legend()

axes[1].plot(n1,'tomato',   lw=2,alpha=0.7,label='broken')
axes[1].plot(n2,'steelblue',lw=2,alpha=0.7,label='lower lr')
axes[1].plot(n3,'orange',   lw=2,alpha=0.7,label='clipped')
axes[1].plot(n4,'seagreen', lw=2,alpha=0.7,label='Adam')
axes[1].set_title('Gradient Norms'); axes[1].set_ylim(0,50); axes[1].legend()
plt.tight_layout(); plt.show()

print('Root cause: lr=1.0 -> gradient steps too large -> weights diverge')
print('Fix 1: lower lr | Fix 2: clip_grad_norm_ | Fix 3: Adam (auto-adapts)')